In [14]:
import openml
from collections import Counter
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from greedy_algorithm.greedy import greedy_submodular_maximization

In [2]:
dataset = openml.datasets.get_dataset("Bioresponse") # Bioresponse dataset

In [3]:
X, y, categorical_indicator, attribute_names = dataset.get_data(
    target=dataset.default_target_attribute,
    dataset_format='dataframe'
)

In [6]:
len(attribute_names)

1776

In [41]:
y.value_counts()

target
1    2034
0    1717
Name: count, dtype: int64

In [46]:
Counter(categorical_indicator)

Counter({False: 1776})

AttributeError: module 'pandas' has no attribute 'counter'

In [ ]:
attribute_names

In [ ]:
y

# Training

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Initializing our budget (how many features we can train on)

In [12]:
k = 50

# Model 1 (all features)

In [10]:
print("\nTraining on ALL features...")
rf_all = RandomForestClassifier(n_estimators=100, random_state=42)
t0 = time.time()
rf_all.fit(X_train, y_train)
t_all = time.time() - t0

preds_all = rf_all.predict(X_test)
acc_all = accuracy_score(y_test, preds_all)
f1_all = f1_score(y_test, preds_all, average='macro')


Training on ALL features...


# Model 2 (random features)

In [13]:
print("\nTraining on RANDOM features...")
np.random.seed(42)
# Randomly select k unique column indices
random_indices = np.random.choice(X_train.shape[1], k, replace=False)

X_train_rand = X_train.iloc[:, random_indices]
X_test_rand = X_test.iloc[:, random_indices]

rf_rand = RandomForestClassifier(n_estimators=100, random_state=42)
t0 = time.time()
rf_rand.fit(X_train_rand, y_train)
t_rand = time.time() - t0

preds_rand = rf_rand.predict(X_test_rand)
acc_rand = accuracy_score(y_test, preds_rand)
f1_rand = f1_score(y_test, preds_rand, average='macro')


Training on RANDOM features...


# Model 3 (based on submodular approach)

In [18]:
print("\nCalculating correlation matrix (on X_train only)...")
corr_matrix = X_train.corr().abs().to_numpy().copy()
np.fill_diagonal(corr_matrix, 1.0) 


Calculating correlation matrix (on X_train only)...


In [26]:
np.isnan(corr_matrix).any()

np.True_

In [27]:
print("\nCalculating correlation matrix (on X_train only)...")
corr_matrix = X_train.corr().abs().fillna(0).to_numpy().copy()
np.fill_diagonal(corr_matrix, 1.0) 


Calculating correlation matrix (on X_train only)...


In [28]:
np.isnan(corr_matrix).any()

np.False_

In [29]:
print("Submodular selection...")
t0 = time.time()
submodular_indices = greedy_submodular_maximization(X_train.shape[1], k, corr_matrix)
t_select = time.time() - t0

Submodular selection...
Starting selection of 50 elements out of 1776...
Step 1: Selected element 504, Marginal Gain: 361.8194
Step 2: Selected element 1738, Marginal Gain: 123.5600
Step 3: Selected element 1336, Marginal Gain: 73.8409
Step 4: Selected element 850, Marginal Gain: 56.7635
Step 5: Selected element 2, Marginal Gain: 37.2594
Step 6: Selected element 448, Marginal Gain: 26.8270
Step 7: Selected element 1175, Marginal Gain: 25.6473
Step 8: Selected element 892, Marginal Gain: 22.6703
Step 9: Selected element 862, Marginal Gain: 21.9409
Step 10: Selected element 658, Marginal Gain: 17.6916
Step 11: Selected element 420, Marginal Gain: 13.3779
Step 12: Selected element 950, Marginal Gain: 10.5359
Step 13: Selected element 166, Marginal Gain: 9.4801
Step 14: Selected element 913, Marginal Gain: 7.8339
Step 15: Selected element 454, Marginal Gain: 7.8231
Step 16: Selected element 190, Marginal Gain: 7.7425
Step 17: Selected element 1604, Marginal Gain: 6.7498
Step 18: Selected e

In [30]:
X_train_sub = X_train.iloc[:, submodular_indices]
X_test_sub = X_test.iloc[:, submodular_indices]

print("Training on SUBMODULAR features...")
rf_sub = RandomForestClassifier(n_estimators=100, random_state=42)
t0 = time.time()
rf_sub.fit(X_train_sub, y_train)
t_sub = time.time() - t0

preds_sub = rf_sub.predict(X_test_sub)
acc_sub = accuracy_score(y_test, preds_sub)
f1_sub = f1_score(y_test, preds_sub, average='macro')

Training on SUBMODULAR features...


# RESULTS

In [31]:
print("\n" + "="*50)
print("FINAL RESULTS:")
print("="*50)
print(f"1. ALL FEATURES ({X.shape[1]}):")
print(f"   Accuracy: {acc_all:.4f} | F1-Score: {f1_all:.4f} | Training time: {t_all:.2f}s")

print(f"\n2. RANDOM FEATURES ({k}):")
print(f"   Accuracy: {acc_rand:.4f} | F1-Score: {f1_rand:.4f} | Training time: {t_rand:.2f}s")

print(f"\n3. SUBMODULAR FEATURES ({k}):")
print(f"   Accuracy: {acc_sub:.4f} | F1-Score: {f1_sub:.4f} | Training time: {t_sub:.2f}s")
print(f"   (Feature selection time: {t_select:.2f}s)")


FINAL RESULTS:
1. ALL FEATURES (1776):
   Accuracy: 0.8003 | F1-Score: 0.7949 | Training time: 1.31s

2. RANDOM FEATURES (50):
   Accuracy: 0.7164 | F1-Score: 0.7079 | Training time: 0.24s

3. SUBMODULAR FEATURES (50):
   Accuracy: 0.7670 | F1-Score: 0.7628 | Training time: 0.29s
   (Feature selection time: 27.09s)
